# LangGraph Chatbot with Memory

## 1. Basic Idea

This code creates a **simple chatbot using LangGraph**.

The flow is:

```text
User Message
     ↓
LangGraph State
     ↓
Chat Node
     ↓
LLM (OpenAI)
     ↓
AI Response
     ↓
State is updated
     ↓
MemorySaver stores the state
```

---

## 2. Imports

```python
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
```

These are used to:

* Create the LangGraph
* Define the state
* Create user messages
* Connect OpenAI LLM
* Load `.env`
* Store conversation memory

---

## 3. Load Environment Variables

```python
load_dotenv()
```

Loads values from the `.env` file.

For example:

```text
OPENAI_API_KEY=your_api_key
```

---

## 4. Create the LLM

```python
llm = ChatOpenAI(model="gpt-4o-mini")
```

This creates the OpenAI LLM that will generate the chatbot's response.

---

# 5. Define the State

```python
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
```

### What is State?

**State = data that moves through the LangGraph.**

Here, our state contains:

```text
ChatState
   ↓
messages
   ↓
conversation history
```

Example:

```python
{
    "messages": [
        HumanMessage("Hi"),
        AIMessage("Hello!")
    ]
}
```

### `add_messages`

```python
Annotated[list[BaseMessage], add_messages]
```

tells LangGraph to **add new messages to the existing message history** instead of replacing the old messages.

---

# 6. Create the Chat Node

```python
def chat_node(state: ChatState):

    messages = state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}
```

The node does 3 things:

### Step 1: Get messages

```python
messages = state["messages"]
```

Gets the conversation from the state.

### Step 2: Send messages to LLM

```python
response = llm.invoke(messages)
```

The LLM reads the messages and generates a response.

### Step 3: Add AI response to state

```python
return {"messages": [response]}
```

The AI response is added to the conversation.

---

# 7. Create the Graph

```python
checkpoint = MemorySaver()

graph = StateGraph(ChatState)
```

### `StateGraph`

Creates the LangGraph workflow.

### `MemorySaver`

Stores checkpoints of the conversation state.

---

# 8. Add Node

```python
graph.add_node("chat_node", chat_node)
```

This adds our `chat_node` to the graph.

```text
chat_node
    ↓
chat_node() function
```

---

# 9. Add Edges

```python
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
```

This defines the flow:

```text
START
  ↓
chat_node
  ↓
END
```

---

# 10. Compile the Graph

```python
chatbot = graph.compile(checkpointer=checkpoint)
```

`compile()` converts the graph into an **executable chatbot**.

Because we pass:

```python
checkpointer=checkpoint
```

the chatbot can save conversation state.

---

# 11. Thread ID

```python
thread_id = "1"
```

A `thread_id` identifies a particular conversation.

Think of it as a:

```text
Conversation ID
```

Example:

```text
Thread 1 → Robin's conversation

Thread 2 → Another conversation
```

Different thread IDs keep conversations separate.

---

# 12. Invoke the Chatbot

```python
response = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="Hello")
        ]
    },
    config={
        "configurable": {
            "thread_id": thread_id
        }
    }
)
```

This sends the user message to the LangGraph.

The graph executes:

```text
User Message
     ↓
START
     ↓
chat_node
     ↓
OpenAI LLM
     ↓
AI Response
     ↓
END
```

---

# 13. Get the AI Response

```python
response["messages"][-1].content
```

`messages` contains the conversation.

```python
[-1]
```

means:

> Get the last message.

`.content` gets the actual text.

Example:

```text
AIMessage("Hello! How can I help you?")
```

becomes:

```text
Hello! How can I help you?
```

---

# 14. How Memory Works

Suppose we use:

```python
thread_id = "1"
```

First message:

```text
User: Hi
AI: Hello!
```

Then:

```text
User: My name is Robin
AI: Nice to meet you Robin!
```

Then:

```text
User: What is my name?
```

Because all these messages use:

```text
thread_id = "1"
```

the conversation history is available.

So the AI can answer:

```text
Your name is Robin.
```

### Important:

```text
MemorySaver
    +
thread_id
    +
add_messages
    ↓
Conversation memory
```

---

# 15. Complete Flow

```text
                USER
                  ↓
          HumanMessage
                  ↓
             ChatState
                  ↓
              chat_node
                  ↓
           ChatOpenAI
                  ↓
             AIMessage
                  ↓
             ChatState
                  ↓
           MemorySaver
                  ↓
             thread_id
```

---

# 16. Easy Way to Remember

Remember these **6 things**:

| Concept       | Meaning                           |
| ------------- | --------------------------------- |
| `StateGraph`  | Creates the workflow              |
| `State`       | Data flowing through the workflow |
| `Node`        | Performs the actual work          |
| `Edge`        | Defines the flow                  |
| `MemorySaver` | Saves/checkpoints state           |
| `thread_id`   | Identifies the conversation       |

### One-line summary

> **User message → State → Node → LLM → AI response → State → Memory**

This is the basic foundation of LangGraph. 🚀
